In [1]:
#演示CNN的综合案例，图像分类。
#回顾：深度学习项目的研发流程：准备数据集（torchvision自带的CIFAR10数据集，包含6w张（32，32，3）的图片，5w张训练集，1w张测试集。10个分类，每个分类6k张
#搭建卷积神经网络 3、模型训练 4、模型测试
#卷积层：提取图像的局部特征 --- 特征图，计算方式 N = math.floor（W-F+2P）//S +1
#池化层：降维，有最大池化和平均池化。

In [4]:
import torch
import torch.nn as nn
from torchvision.datasets import CIFAR10
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import time 
import matplotlib.pyplot as plt


In [7]:
#每批次样本数
batch_size = 8

#准备数据集
def create_dataset():
    #1、获取训练集
    train_dataset = CIFAR10(root = './data',train = True,transform = ToTensor(),download= True)
    test_dataset = CIFAR10(root = './data',train = False,transform= ToTensor(),download = True)
    return  train_dataset,test_dataset

In [ ]:
#图像展示(第12张图片)
plt.figure(figsize = (2,2))
plt.imshow(train_dataset.data[11])
plt.title(train_dataset.targets[11])
plt.show()

In [ ]:
#构建卷积神经网络
class ImageModel(nn.Module):
    def __init__(self):
        super().__init__()
        #搭建神网络
        #第一个卷积层,输入3通道，输出6通道
        self.conv1 = nn.Conv2d(in_channels =3,out_channels = 6,kernel_size = 3,padding=0,stride=1)
        #第一个池化层
        self.pool1 = nn.MaxPool2d(kernel_size=2,stride=2,padding=0)
        #第2个卷积层
        self.conv2 = nn.Conv2d(in_channels = 6,out_channels=16,kernel_size=3)
        #第2个池化层
        self.pool2 =nn.MaxPool2d(kernel_size=2,stride=2,padding=0)
        #全连接层
        self.linear1= nn.Linear(in_features=576,ou_features=120)
        self.linear2= nn.Linear(in_features=120,ou_features=84)
        #输出：10类
        self.output= nn.Linear(in_features=84,ou_features=10)
        #注意：这里不需要加入softmax，因为后续用crossentropy计算损失时，softmax的处理包含在其中了。

        def forward(self,x):
            #第1层，卷积层（加权求和）+激活+池化层
            x = self.conv1(x)  #卷积处理
            x = torch.relu(x)  #激活层
            x = self.pool1(x)  #池化层
            #第2层
            x = self.conv2(x)  #卷积处理
            x = torch.relu(x)  #激活层
            x = self.pool2(x)  #池化层
            #全连接层
            #先将数据进行拉平成二维的。（8，16，6，6） → （8，576）
            #参1：样本数（行数），参2，列数（特征数），-1表示自动计算
            x = x.reshape(x.size(0),-1)
            x = torch.relu(self.linear1(x))
            x = torch.relu(self.linear2(x))
            return self.output(x)
model = ImageModel()
#卷积核参数计算公式 = 输入通道数*卷积核尺寸 * 卷积核数量 + 卷积核数量
            

In [ ]:
def train(train_dataset):
    dataloader = DataLoader(train_dataset,batch_size = batch_size,shuffle=True)
    model = ImageModel()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(),lr =1e-3)
    epoch=10
    #循环遍历epoch，开始每轮的训练动作
    for epoch_idx in range(epoch):
        #定义变量，记录总损失，总样本数据量，预测正确样本个数
        total_loss,samples,total_correct,start = 0.0,0,0,time.time()
        #遍历数据加载器，获取到每批次的数据
        for x,y in dataloader:
            #切换训练模式
            model.train()
            #模型预测
            y_pred = model(x)
            #计算损失
            loss = criterion(y_pred,y)
            #梯度清零，反向传播，参数更新
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            #记录预测正确的样本数量
            print(torch.argmax(y_pred,dim = -1))  #argmax返回每张图对应的最大概率值对应的索引（属于第几类）
            print(y)  #真实分类
            print(torch.argmax(y_pred,dim=--1)==y.sum()) #预测正确的样本数
            total_correct += (torch.argmax(y_pred,dim=--1)==y).sum()
            #统计当前批次的总损失
            total_loss += loss.item() * len(y)   #
            #统计当前批次的总样本个数
            total_samples += len(y)
        #一轮训练结束,打印该轮的训练信息
        print(f'epoch:{epoch + 1}，loss：{total_loss/total_samples:.5f},acc:{total_correct/total_samples:.2f}')
        
        

In [ ]:
#模型测试
def evaluate(test_dataset):
    #创建测试集，数据加载器
    dataloader = DataLoader(test_dataset,batch_size = batch_size,shuffle=True)
    #创建模型对象
    model = ImageModel()
    #加载模型参数
    model.load_state_dict(torch.load('./model/image_model.pth'))
    total_correct,total_sample = 0,0
    for x,y in dataloader:
        #切换模型模式
        model.eval()
        #模型预测
        y_pred = model(x)
        y_pred = torch.argmax(y_pred,dim=-1)
        total_correct+=(y_pred == y).sum()
        total_samples += len(y)
    print(f'{total_correct/total_samples}:.2f')